# K-Means Clustering

## Using code from Lab 10: Discovering Player Archetypes

Goal:
Identify meaningful borrower archetypes in the loan dataset.

Questions:
- Are there distinct groups of borrowers?
- Which borrower groups have higher default rates?
- Do clusters reveal patterns not captured by supervised models?

All the imports

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

Loading the data

In [2]:
loan_df_clean = pd.read_csv("Loan_default_cleaned.csv")

loan_df_clean.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DebtToIncomeRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default,LoanToIncomeRatio
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0,0.588262
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0,2.467481
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1,1.534154
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0,1.412638
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0,0.447179


In [7]:
loan_df_clean.shape

(255347, 18)

In [3]:
loan_sample = loan_df_clean.sample(
    n=10000,
    random_state=6021
).copy()

In [4]:
cluster_vars = [
    'Age',
    'Income',
    'LoanAmount',
    'CreditScore',
    'MonthsEmployed',
    'InterestRate',
    'DebtToIncomeRatio',
    'LoanToIncomeRatio'
]

In [8]:
X_raw = loan_sample[cluster_vars]
X_raw.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,InterestRate,DebtToIncomeRatio,LoanToIncomeRatio
54415,37,73592,164134,328,47,11.68,0.65,2.230324
109645,46,44598,105239,741,41,23.52,0.72,2.359725
105324,43,115229,45773,797,91,22.32,0.25,0.397235
45591,31,109962,157949,336,81,23.41,0.54,1.436396
62527,18,47272,159788,779,31,17.70,0.38,3.380183


In [9]:
# Standardize variables before clustering

scaler = StandardScaler()

X_z = scaler.fit_transform(X_raw)

print(X_z.shape)

(10000, 8)


In [10]:
summary = (
    X_raw
    .agg(['mean', 'std', 'min', 'max'])
    .T
    .round(2)
)

display(summary)

,mean,std,min,max
Age,43.63,15.07,18.00,69.00
Income,82228.65,38856.74,15008.00,149999.00
LoanAmount,127398.46,70626.51,5006.00,249989.00
CreditScore,573.90,159.53,300.00,849.00
MonthsEmployed,59.10,34.71,0.00,119.00
InterestRate,13.47,6.64,2.00,25.00
DebtToIncomeRatio,0.49,0.23,0.10,0.90
LoanToIncomeRatio,2.16,2.11,0.04,16.02


In [11]:
results = []

for k in range(2, 9):

    model = KMeans(
        n_clusters=k,
        random_state=6021,
        n_init=20
    )

    model.fit(X_z)

    results.append({
        "K": k,
        "WCSS": model.inertia_,
        "Silhouette": silhouette_score(
            X_z,
            model.labels_,
            sample_size=5000,
            random_state=6021
        )
    })

k_scores = pd.DataFrame(results)

display(k_scores.round(3))

: 